# 자연어 처리를 위한 Transformer 구현 및 감성분석 적용 실습

이 노트북은 **Self-Attention → Transformer Block → Word/Positional Embedding → 감성분석 분류기** 순서로 구성하였다.

핵심은 다음과 같다.

- Transformer Block은 **Self-Attention, Residual Connection, Layer Normalization, Feed Forward Neural Network**로 구성된다.
- Self-Attention은 입력 토큰에서 **Query, Key, Value**를 만들고, `QK^T` 점수를 계산한 뒤 `softmax`로 중요도를 만든다.
- `sqrt(d_k)`로 나누는 이유는 내적값이 지나치게 커져 softmax가 한쪽으로 쏠리는 문제와 학습 불안정을 줄이기 위해서이다.
- Transformer는 RNN을 사용하지 않으므로 토큰의 순서를 알려 주기 위해 **Positional Embedding**이 필요하다.
- 문장 분류에서는 Transformer 출력 전체를 평균 풀링한 뒤 분류층을 통과시켜 긍정/부정을 예측한다.


## 1. 실습 환경 설치

PyTorch 기반 Transformer 구현을 다룬다. 이 코드에서는 학습 루프를 간결하게 작성하기 위해 `pytorch-lightning`을 사용하고, 정확도 계산을 위해 `torchmetrics`를 사용한다. Colab 환경에서 처음 실행하는 경우 필요한 패키지를 설치한다.


In [1]:
# Colab 또는 새 가상환경에서 PyTorch Lightning과 TorchMetrics가 없을 수 있으므로 설치합니다.
# -q 옵션은 설치 로그를 간단히 출력하게 하여 노트북 화면을 깔끔하게 유지합니다.
!pip install -q pytorch-lightning torchmetrics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.6 MB/s eta 0:00:00


## 2. 필요한 라이브러리 불러오기

Self-Attention과 Transformer는 행렬 연산이 핵심이다. `torch`는 텐서 계산, `torch.nn`은 신경망 계층, `torch.nn.functional`은 softmax 같은 함수형 연산에 사용한다. `pytorch_lightning`은 학습/검증/테스트 루프를 관리한다.


In [2]:
# 수치 계산과 배열 처리를 위해 NumPy를 불러옵니다.
import numpy as np

# PyTorch의 핵심 패키지인 torch를 불러옵니다.
import torch

# PyTorch 신경망 계층을 만들기 위해 torch.nn 모듈을 nn이라는 별칭으로 불러옵니다.
from torch import nn

# softmax와 같은 함수형 신경망 연산을 사용하기 위해 torch.nn.functional을 F라는 별칭으로 불러옵니다.
import torch.nn.functional as F

# 여러 개의 텐서를 하나의 데이터셋으로 묶기 위해 TensorDataset을 불러옵니다.
from torch.utils.data import TensorDataset

# 미니배치 단위로 데이터를 모델에 공급하기 위해 DataLoader를 불러옵니다.
from torch.utils.data import DataLoader

# 모델 파라미터를 업데이트하는 Adam 최적화 함수를 불러옵니다.
from torch.optim import Adam

# 학습 과정을 단순화하기 위해 PyTorch Lightning을 pl이라는 별칭으로 불러옵니다.
import pytorch_lightning as pl

# 분류 정확도를 계산하기 위해 torchmetrics의 Accuracy 클래스를 불러옵니다.
from torchmetrics.classification import Accuracy

# IMDB 영화 리뷰 데이터셋을 불러오기 위해 TensorFlow Keras의 datasets 모듈을 사용합니다.
from tensorflow.keras.datasets import imdb

# 길이가 서로 다른 문장 시퀀스를 같은 길이로 맞추기 위해 pad_sequences를 사용합니다.
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 실험 결과가 가능한 한 동일하게 나오도록 난수 시드를 고정합니다.
pl.seed_everything(42, workers=True)


INFO:lightning_fabric.utilities.seed:Seed set to 42


42

## 3. Self-Attention 구현

Self-Attention은 입력 문장의 각 토큰이 다른 토큰과 얼마나 관련 있는지 계산한다. 입력 임베딩 `x`에서 Query, Key, Value를 각각 만들고, `QK^T / sqrt(d_k)`를 계산한 뒤 softmax를 적용하여 Attention 가중치를 만든다. 마지막으로 이 가중치를 Value에 곱해 문맥이 반영된 표현을 얻는다.


In [3]:
# SelfAttention 클래스는 PyTorch의 모든 신경망 모듈이 상속하는 nn.Module을 상속합니다.
class SelfAttention(nn.Module):
    # __init__ 메서드는 SelfAttention 계층에서 사용할 선형 변환 계층들을 초기화합니다.
    def __init__(self, d: int, heads: int = 8):
        # 부모 클래스인 nn.Module의 초기화 기능을 실행합니다.
        super().__init__()

        # d는 한 토큰을 표현하는 임베딩 벡터의 차원 수입니다.
        self.d = d

        # heads는 Multi-Head Attention에서 병렬로 사용할 attention head의 개수입니다.
        self.h = heads

        # 입력 벡터를 Query 벡터로 변환하는 선형 계층입니다.
        # 출력 차원을 d * heads로 만들어 head별 Query를 한 번에 계산합니다.
        self.WQ = nn.Linear(d, d * heads, bias=False)

        # 입력 벡터를 Key 벡터로 변환하는 선형 계층입니다.
        # Key는 Query와 내적되어 토큰 간 관련도 점수를 계산하는 데 사용됩니다.
        self.WK = nn.Linear(d, d * heads, bias=False)

        # 입력 벡터를 Value 벡터로 변환하는 선형 계층입니다.
        # Value는 attention 가중치가 곱해져 최종 문맥 벡터를 만드는 데 사용됩니다.
        self.WV = nn.Linear(d, d * heads, bias=False)

        # 여러 head에서 나온 결과를 다시 하나의 d차원 벡터로 합치는 선형 계층입니다.
        self.unifyheads = nn.Linear(heads * d, d)

    # forward 메서드는 입력 텐서 x를 받아 Self-Attention 결과를 반환합니다.
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x의 크기는 (배치 크기, 문장 길이, 임베딩 차원)입니다.
        b, l, d = x.size()

        # 현재 attention head 개수를 지역 변수 h에 저장하여 코드 가독성을 높입니다.
        h = self.h

        # 입력 x를 Query로 변환한 뒤 (b, l, h, d) 형태로 바꿉니다.
        # transpose(1, 2)는 문장 길이 축과 head 축을 바꾸어 head별 계산을 쉽게 만듭니다.
        # contiguous()는 transpose 후 메모리 배치를 연속적으로 만들어 view가 안전하게 동작하도록 합니다.
        # 최종적으로 (b*h, l, d) 형태로 만들어 각 head를 독립 배치처럼 계산합니다.
        queries = self.WQ(x).view(b, l, h, d).transpose(1, 2).contiguous().view(b * h, l, d)

        # 입력 x를 Key로 변환하고 Query와 동일한 방식으로 (b*h, l, d) 형태로 정리합니다.
        keys = self.WK(x).view(b, l, h, d).transpose(1, 2).contiguous().view(b * h, l, d)

        # 입력 x를 Value로 변환하고 Query와 동일한 방식으로 (b*h, l, d) 형태로 정리합니다.
        values = self.WV(x).view(b, l, h, d).transpose(1, 2).contiguous().view(b * h, l, d)

        # Query와 Key의 전치 행렬을 배치 행렬곱으로 계산하여 토큰 간 관련도 점수를 구합니다.
        # np.sqrt(d)로 나누어 점수 크기를 안정화하고 softmax가 한쪽으로 과도하게 쏠리는 문제를 줄입니다.
        attention_scores = torch.bmm(queries, keys.transpose(1, 2)) / np.sqrt(d)

        # 마지막 차원 기준으로 softmax를 적용하여 각 토큰이 다른 토큰을 얼마나 참고할지 확률 형태로 만듭니다.
        attention_weights = F.softmax(attention_scores, dim=-1)

        # attention 가중치를 Value에 곱하여 문맥 정보가 반영된 토큰 표현을 계산합니다.
        out = torch.bmm(attention_weights, values).view(b, h, l, d)

        # head 축과 문장 길이 축을 다시 원래 순서에 가깝게 바꿉니다.
        # 이후 모든 head의 결과를 이어 붙여 (b, l, h*d) 형태로 만듭니다.
        out = out.transpose(1, 2).contiguous().view(b, l, h * d)

        # 이어 붙인 multi-head 결과를 선형 계층에 통과시켜 다시 d차원 표현으로 통합합니다.
        return self.unifyheads(out)


## 4. Transformer Block 구현

Transformer Block은 Self-Attention 결과를 입력과 더하는 Residual Connection을 사용하고, 그 결과를 Layer Normalization으로 안정화한다. 이후 Feed Forward Neural Network를 통과시키고 다시 Residual Connection과 Layer Normalization을 적용한다.


In [4]:
# TransformerBlock 클래스는 하나의 Transformer 인코더 블록을 구현합니다.
class TransformerBlock(nn.Module):
    # __init__ 메서드는 Self-Attention, LayerNorm, FFNN 계층을 초기화합니다.
    def __init__(self, d: int, heads: int = 8, n_mlp: int = 4):
        # 부모 클래스인 nn.Module의 초기화 기능을 실행합니다.
        super().__init__()

        # SelfAttention 모듈을 생성하여 입력 토큰 간 문맥 관계를 계산하도록 합니다.
        self.attention = SelfAttention(d, heads=heads)

        # 첫 번째 LayerNorm은 attention 결과와 원래 입력을 더한 뒤 분포를 안정화합니다.
        self.norm1 = nn.LayerNorm(d)

        # 두 번째 LayerNorm은 FFNN 결과와 이전 입력을 더한 뒤 분포를 안정화합니다.
        self.norm2 = nn.LayerNorm(d)

        # Feed Forward Neural Network는 각 토큰 위치마다 독립적으로 적용되는 작은 완전연결 신경망입니다.
        self.ff = nn.Sequential(
            # 첫 번째 선형 계층은 d차원 표현을 n_mlp*d차원으로 확장하여 더 풍부한 특징을 학습합니다.
            nn.Linear(d, n_mlp * d),

            # ReLU 활성화 함수는 비선형성을 추가하여 복잡한 패턴을 학습하게 합니다.
            nn.ReLU(),

            # 두 번째 선형 계층은 확장된 특징을 다시 d차원으로 축소하여 블록 출력 크기를 입력과 맞춥니다.
            nn.Linear(n_mlp * d, d),
        )

    # forward 메서드는 입력 x를 Transformer Block에 통과시킵니다.
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Self-Attention을 적용하여 각 토큰이 문장 안의 다른 토큰을 참고한 표현을 만듭니다.
        x_prime = self.attention(x)

        # Residual Connection으로 attention 결과와 원래 입력을 더하고 LayerNorm을 적용합니다.
        x = self.norm1(x_prime + x)

        # Feed Forward Neural Network를 적용하여 토큰별 표현을 추가로 변환합니다.
        x_prime = self.ff(x)

        # FFNN 결과와 입력을 다시 더한 뒤 LayerNorm을 적용하여 최종 블록 출력을 반환합니다.
        return self.norm2(x_prime + x)


## 5. IMDB 감성분석 데이터 준비

감성분석 적용 단계에 해당한다. IMDB 데이터셋은 영화 리뷰 문장을 정수 인덱스 시퀀스로 제공한다. Transformer에 입력하려면 모든 문장을 같은 길이로 맞추어야 하므로 padding을 적용한다.


In [5]:
# IMDBDataLoader 클래스는 PyTorch Lightning의 DataModule을 상속하여 데이터 준비 과정을 모듈화합니다.
class IMDBDataLoader(pl.LightningDataModule):
    # __init__ 메서드는 데이터 로딩에 필요한 하이퍼파라미터를 저장합니다.
    def __init__(self, batch_size: int, num_words: int, max_seq_len: int, train_samples: int | None = 5000, test_samples: int | None = 2000):
        # 부모 클래스인 LightningDataModule의 초기화 기능을 실행합니다.
        super().__init__()

        # 한 번의 학습 단계에서 사용할 샘플 개수인 배치 크기를 저장합니다.
        self.batch_size = batch_size

        # 사용할 단어 사전 크기를 저장합니다.
        self.num_words = num_words

        # 각 리뷰 문장을 맞출 최대 시퀀스 길이를 저장합니다.
        self.max_seq_len = max_seq_len

        # 실습 시간을 줄이기 위해 사용할 학습 샘플 개수를 저장합니다.
        # 전체 데이터를 사용하려면 None으로 설정하면 됩니다.
        self.train_samples = train_samples

        # 실습 시간을 줄이기 위해 사용할 테스트 샘플 개수를 저장합니다.
        # 전체 데이터를 사용하려면 None으로 설정하면 됩니다.
        self.test_samples = test_samples

    # setup 메서드는 실제 데이터를 다운로드하고 전처리합니다.
    def setup(self, stage: str | None = None):
        # Keras에서 제공하는 IMDB 영화 리뷰 데이터를 불러옵니다.
        # num_words는 빈도 상위 num_words개 단어만 사용하도록 제한합니다.
        (x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=self.num_words)

        # 실습 속도를 높이기 위해 지정된 개수만큼 학습 데이터를 잘라 사용합니다.
        if self.train_samples is not None:
            x_train = x_train[:self.train_samples]
            y_train = y_train[:self.train_samples]

        # 실습 속도를 높이기 위해 지정된 개수만큼 테스트 데이터를 잘라 사용합니다.
        if self.test_samples is not None:
            x_test = x_test[:self.test_samples]
            y_test = y_test[:self.test_samples]

        # 단어에서 정수 인덱스로 가는 사전을 불러옵니다.
        raw_word2idx = imdb.get_word_index()

        # Keras IMDB 데이터는 0, 1, 2, 3번 인덱스를 특수 토큰으로 사용하므로 기존 인덱스에 3을 더합니다.
        self.word2idx = {word: index + 3 for word, index in raw_word2idx.items()}

        # padding, 시작, 알 수 없는 단어, 미사용 토큰에 해당하는 특수 토큰을 사전에 추가합니다.
        self.word2idx.update({'<PAD>': 0, '<START>': 1, '<UNK>': 2, '<UNUSED>': 3})

        # 정수 인덱스를 다시 단어로 바꾸기 위해 역방향 사전을 만듭니다.
        self.idx2word = {index: word for word, index in self.word2idx.items()}

        # 학습 리뷰의 길이를 max_seq_len으로 맞춥니다.
        # 짧은 문장은 앞쪽에 0을 채우고, 긴 문장은 앞쪽을 잘라 마지막 max_seq_len개 토큰만 사용합니다.
        x_train = pad_sequences(x_train, maxlen=self.max_seq_len, value=0, padding='pre', truncating='pre')

        # 테스트 리뷰도 학습 데이터와 동일한 방식으로 길이를 맞춥니다.
        x_test = pad_sequences(x_test, maxlen=self.max_seq_len, value=0, padding='pre', truncating='pre')

        # NumPy 배열을 PyTorch LongTensor로 변환하여 임베딩 계층에 입력할 수 있게 합니다.
        self.x_train = torch.LongTensor(x_train)

        # 정답 레이블도 CrossEntropyLoss가 요구하는 LongTensor 형태로 변환합니다.
        self.y_train = torch.LongTensor(y_train)

        # 테스트 입력도 PyTorch LongTensor로 변환합니다.
        self.x_test = torch.LongTensor(x_test)

        # 테스트 정답도 PyTorch LongTensor로 변환합니다.
        self.y_test = torch.LongTensor(y_test)

    # example 메서드는 전처리된 정수 시퀀스를 사람이 읽을 수 있는 리뷰 문장으로 복원해 확인합니다.
    def example(self) -> str:
        # 학습 데이터 중 하나의 인덱스를 무작위로 선택합니다.
        idx = np.random.randint(0, len(self.x_train))

        # 선택한 인덱스에 해당하는 입력 시퀀스와 정답 레이블을 가져옵니다.
        x, y = self.x_train[idx], self.y_train[idx]

        # padding 토큰과 특수 토큰을 제외하고 정수 인덱스를 단어로 변환합니다.
        review = ' '.join(self.idx2word.get(int(token_id), '<UNK>') for token_id in x if int(token_id) > 3)

        # 레이블이 1이면 긍정, 0이면 부정으로 표시합니다.
        sentiment = 'POSITIVE' if int(y) == 1 else 'NEGATIVE'

        # 복원된 리뷰와 감성 레이블을 문자열로 반환합니다.
        return f'Review : {review}\nSentiment: {sentiment}'

    # train_dataloader 메서드는 학습용 DataLoader를 반환합니다.
    def train_dataloader(self):
        # 학습 입력과 정답을 하나의 TensorDataset으로 묶습니다.
        dataset = TensorDataset(self.x_train, self.y_train)

        # DataLoader를 만들어 미니배치 단위로 데이터를 섞어서 공급합니다.
        return DataLoader(dataset, batch_size=self.batch_size, shuffle=True, num_workers=2, persistent_workers=True)

    # val_dataloader 메서드는 검증용 DataLoader를 반환합니다.
    def val_dataloader(self):
        # 테스트 입력과 정답을 검증 데이터로 사용하기 위해 TensorDataset으로 묶습니다.
        dataset = TensorDataset(self.x_test, self.y_test)

        # 검증 단계에서는 결과 비교를 위해 데이터를 섞지 않습니다.
        return DataLoader(dataset, batch_size=self.batch_size, shuffle=False, num_workers=2, persistent_workers=True)

    # test_dataloader 메서드는 테스트용 DataLoader를 반환합니다.
    def test_dataloader(self):
        # 테스트 입력과 정답을 TensorDataset으로 묶습니다.
        dataset = TensorDataset(self.x_test, self.y_test)

        # 테스트 단계에서도 데이터를 섞지 않고 순서대로 평가합니다.
        return DataLoader(dataset, batch_size=self.batch_size, shuffle=False, num_workers=2, persistent_workers=True)


## 6. Transformer 감성분석 모델 구현

전체 구조를 코드로 구현한다. 입력 정수 시퀀스는 Word Embedding을 통해 단어 의미 벡터로 바뀌고, Positional Embedding이 더해져 토큰 순서 정보가 반영된다. 이후 여러 개의 Transformer Block을 통과하고, 전체 토큰 출력의 평균을 내어 문장 하나의 벡터로 만든 뒤 분류층에서 긍정/부정을 예측한다.


In [6]:
# IMDBTransformer 클래스는 IMDB 감성분석을 위한 Transformer 기반 분류 모델입니다.
class IMDBTransformer(pl.LightningModule):
    # __init__ 메서드는 모델 구조와 학습 설정을 초기화합니다.
    def __init__(
        self,
        d: int = 128,
        heads: int = 8,
        depth: int = 4,
        max_seq_len: int = 128,
        num_tokens: int = 10000,
        num_classes: int = 2,
        learning_rate: float = 1e-4,
    ):
        # 부모 클래스인 LightningModule의 초기화 기능을 실행합니다.
        super().__init__()

        # 전달받은 하이퍼파라미터를 self.hparams에 자동 저장하여 로그와 체크포인트에서 확인할 수 있게 합니다.
        self.save_hyperparameters()

        # 사용할 단어 사전 크기를 저장합니다.
        self.num_tokens = num_tokens

        # 정수 토큰 ID를 d차원의 단어 임베딩 벡터로 변환하는 계층입니다.
        self.token_emb = nn.Embedding(num_tokens, d)

        # 각 위치 인덱스를 d차원의 위치 임베딩 벡터로 변환하는 계층입니다.
        # Transformer는 RNN처럼 순서대로 처리하지 않기 때문에 위치 정보가 반드시 필요합니다.
        self.pos_emb = nn.Embedding(max_seq_len, d)

        # 여러 개의 TransformerBlock을 순서대로 쌓아 깊은 Transformer 모델을 구성합니다.
        self.transformer_blocks = nn.Sequential(
            # depth 개수만큼 TransformerBlock을 생성하여 리스트로 만든 뒤 nn.Sequential에 전달합니다.
            *[TransformerBlock(d=d, heads=heads) for _ in range(depth)]
        )

        # 평균 풀링으로 만든 문장 벡터를 긍정/부정 클래스 점수로 변환하는 선형 분류층입니다.
        self.classification = nn.Linear(d, num_classes)

        # 다중 클래스 분류 손실 함수입니다.
        # 이진 분류도 클래스가 2개인 다중 클래스 문제로 처리할 수 있습니다.
        self.criterion = nn.CrossEntropyLoss()

        # 학습 정확도를 계산하는 TorchMetrics 객체입니다.
        self.train_accuracy = Accuracy(task='multiclass', num_classes=num_classes)

        # 검증 정확도를 계산하는 TorchMetrics 객체입니다.
        self.val_accuracy = Accuracy(task='multiclass', num_classes=num_classes)

        # 테스트 정확도를 계산하는 TorchMetrics 객체입니다.
        self.test_accuracy = Accuracy(task='multiclass', num_classes=num_classes)

    # forward 메서드는 입력 토큰 시퀀스를 받아 클래스별 점수 logits를 반환합니다.
    def forward(self, x: torch.LongTensor) -> torch.FloatTensor:
        # x의 크기는 (배치 크기, 문장 길이)입니다.
        b, l = x.size()

        # 임베딩 차원 d를 하이퍼파라미터에서 가져옵니다.
        d = self.hparams.d

        # 입력 정수 토큰 ID를 단어 임베딩 벡터로 변환합니다.
        tokens = self.token_emb(x)

        # 0부터 문장 길이 l-1까지의 위치 인덱스를 생성합니다.
        position_ids = torch.arange(l, device=self.device)

        # 위치 인덱스를 위치 임베딩 벡터로 변환합니다.
        positions = self.pos_emb(position_ids)

        # 위치 임베딩을 배치 크기만큼 확장하여 tokens와 같은 크기인 (b, l, d)로 만듭니다.
        positions = positions.unsqueeze(0).expand(b, l, d)

        # 단어 임베딩과 위치 임베딩을 더하여 의미 정보와 순서 정보를 모두 가진 입력 표현을 만듭니다.
        embeddings = tokens + positions

        # 임베딩 표현을 여러 개의 Transformer Block에 통과시켜 문맥이 반영된 토큰 표현을 얻습니다.
        out = self.transformer_blocks(embeddings)

        # 문장 길이 차원(dim=1)을 평균 내어 토큰별 출력들을 하나의 문장 벡터로 요약합니다.
        out = out.mean(dim=1)

        # 문장 벡터를 분류층에 통과시켜 각 클래스에 대한 점수 logits를 계산합니다.
        out = self.classification(out)

        # softmax 전의 클래스 점수 logits를 반환합니다.
        return out

    # configure_optimizers 메서드는 학습에 사용할 최적화 함수를 정의합니다.
    def configure_optimizers(self):
        # Adam은 학습률을 적응적으로 조절하여 신경망 학습에 널리 사용되는 최적화 함수입니다.
        return Adam(self.parameters(), lr=self.hparams.learning_rate)

    # training_step 메서드는 학습 배치 하나에 대한 손실과 정확도를 계산합니다.
    def training_step(self, batch, batch_idx):
        # batch에서 입력 x와 정답 y를 분리합니다.
        x, y = batch

        # 모델에 입력 x를 넣어 클래스별 점수 logits를 계산합니다.
        logits = self(x)

        # logits와 정답 y를 비교하여 CrossEntropyLoss를 계산합니다.
        loss = self.criterion(logits, y)

        # logits에서 가장 큰 점수를 가진 클래스 인덱스를 예측값으로 선택합니다.
        preds = torch.argmax(logits, dim=1)

        # 예측값과 정답을 비교하여 학습 정확도를 계산합니다.
        acc = self.train_accuracy(preds, y)

        # 학습 손실을 진행 막대와 로그에 기록합니다.
        self.log('train_loss', loss, on_step=False, on_epoch=True, prog_bar=True)

        # 학습 정확도를 진행 막대와 로그에 기록합니다.
        self.log('train_acc', acc, on_step=False, on_epoch=True, prog_bar=True)

        # PyTorch Lightning이 역전파에 사용할 손실값을 반환합니다.
        return loss

    # validation_step 메서드는 검증 배치 하나에 대한 손실과 정확도를 계산합니다.
    def validation_step(self, batch, batch_idx):
        # batch에서 입력 x와 정답 y를 분리합니다.
        x, y = batch

        # 모델에 입력 x를 넣어 클래스별 점수 logits를 계산합니다.
        logits = self(x)

        # logits와 정답 y를 비교하여 검증 손실을 계산합니다.
        loss = self.criterion(logits, y)

        # 가장 높은 점수의 클래스를 검증 예측값으로 선택합니다.
        preds = torch.argmax(logits, dim=1)

        # 검증 예측값과 정답을 비교하여 정확도를 계산합니다.
        acc = self.val_accuracy(preds, y)

        # 검증 손실을 진행 막대와 로그에 기록합니다.
        self.log('val_loss', loss, on_step=False, on_epoch=True, prog_bar=True)

        # 검증 정확도를 진행 막대와 로그에 기록합니다.
        self.log('val_acc', acc, on_step=False, on_epoch=True, prog_bar=True)

        # 검증 손실을 반환합니다.
        return loss

    # test_step 메서드는 테스트 배치 하나에 대한 손실과 정확도를 계산합니다.
    def test_step(self, batch, batch_idx):
        # batch에서 입력 x와 정답 y를 분리합니다.
        x, y = batch

        # 모델에 입력 x를 넣어 클래스별 점수 logits를 계산합니다.
        logits = self(x)

        # logits와 정답 y를 비교하여 테스트 손실을 계산합니다.
        loss = self.criterion(logits, y)

        # 가장 높은 점수의 클래스를 테스트 예측값으로 선택합니다.
        preds = torch.argmax(logits, dim=1)

        # 테스트 예측값과 정답을 비교하여 정확도를 계산합니다.
        acc = self.test_accuracy(preds, y)

        # 테스트 손실을 로그에 기록합니다.
        self.log('test_loss', loss, on_step=False, on_epoch=True)

        # 테스트 정확도를 진행 막대와 로그에 기록합니다.
        self.log('test_acc', acc, on_step=False, on_epoch=True, prog_bar=True)

        # 테스트 손실을 반환합니다.
        return loss


## 7. 하이퍼파라미터 설정 및 데이터 확인

모델 학습 전에 단어 사전 크기, 최대 문장 길이, 임베딩 차원, 배치 크기, epoch 수를 설정한다. `imdb_data.example()`을 실행하면 정수 토큰 시퀀스가 실제 단어로 어떻게 복원되는지 확인할 수 있다.


In [7]:
# 사용할 단어 사전 크기를 지정합니다.
NUM_WORDS = 10000

# 각 리뷰 문장을 최대 128개 토큰으로 맞춥니다.
MAX_SEQ_LEN = 128

# 각 토큰을 128차원 임베딩 벡터로 표현합니다.
EMBEDDING_DIM = 128

# 한 번에 32개 리뷰를 학습에 사용합니다.
BATCH_SIZE = 32

# 실습 시간을 고려하여 학습 데이터 일부만 사용합니다.
# 전체 데이터를 사용하려면 TRAIN_SAMPLES = None으로 바꾸면 됩니다.
TRAIN_SAMPLES = 5000

# 실습 시간을 고려하여 테스트 데이터 일부만 사용합니다.
# 전체 데이터를 사용하려면 TEST_SAMPLES = None으로 바꾸면 됩니다.
TEST_SAMPLES = 2000

# 최대 학습 epoch 수를 지정합니다.
MAX_EPOCHS = 5

# IMDBDataLoader 객체를 생성하여 데이터 준비 설정을 저장합니다.
imdb_data = IMDBDataLoader(
    batch_size=BATCH_SIZE,
    num_words=NUM_WORDS,
    max_seq_len=MAX_SEQ_LEN,
    train_samples=TRAIN_SAMPLES,
    test_samples=TEST_SAMPLES,
)

# setup을 직접 호출하여 데이터를 다운로드하고 전처리합니다.
imdb_data.setup()

# 전처리된 리뷰 하나를 단어 문장으로 복원하여 데이터가 정상적으로 준비되었는지 확인합니다.
print(imdb_data.example())


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Review : of course call america's jerry springer i'd talk about the rest of the film but even thinking about the film now is giving me a headache jamie who plays the daughter looks totally unattractive in the movie and remember michael the kick ass karate master from the american ninja series well take a look at him now as a white trash drunk the thing is he really looks too horrible and out of shape to call it getting in touch with his but if your idea of fun is seeing jerry springer sing a country song about his own show or guys up with well just watch the show instead at least steve was smart enough to stay out of this flick
Sentiment: NEGATIVE


## 8. 모델 생성 및 학습 실행

Transformer 감성분석 모델을 생성한 뒤 PyTorch Lightning의 `Trainer`로 학습한다. Colab에서 GPU 런타임을 사용하면 자동으로 GPU를 활용하고, GPU가 없으면 CPU로 실행된다.


In [8]:
# IMDB 감성분석용 Transformer 모델 객체를 생성합니다.
model = IMDBTransformer(
    d=EMBEDDING_DIM,
    heads=8,
    depth=4,
    max_seq_len=MAX_SEQ_LEN,
    num_tokens=NUM_WORDS,
    num_classes=2,
    learning_rate=1e-4,
)

# GPU가 있으면 GPU를 사용하고, 없으면 CPU를 사용하도록 accelerator='auto'로 설정합니다.
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='auto',
    devices='auto',
    log_every_n_steps=10,
)

# trainer.fit은 학습 데이터로 모델을 훈련하고 검증 데이터로 성능을 확인합니다.
trainer.fit(model, datamodule=imdb_data)


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


┏━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name               ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ token_emb          │ Embedding          │  1.3 M │ train │     0 │
│ 1 │ pos_emb            │ Embedding          │ 16.4 K │ train │     0 │
│ 2 │ transformer_blocks │ Sequential         │  2.6 M │ train │     0 │
│ 3 │ classification     │ Linear             │    258 │ train │     0 │
│ 4 │ criterion          │ CrossEntropyLoss   │      0 │ train │     0 │
│ 5 │ train_accuracy     │ MulticlassAccuracy │      0 │ train │     0 │
│ 6 │ val_accuracy       │ MulticlassAccuracy │      0 │ train │     0 │
│ 7 │ test_accuracy      │ MulticlassAccuracy │      0 │ train │     0 │
└───┴────────────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 3.9 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.9 M                                                                                                
Total estimated model params size (MB): 15.693                                                                     
Modules in train mode: 56                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


## 9. 테스트 평가

학습이 끝난 모델을 테스트 데이터에 적용하여 손실과 정확도를 확인한다. 이 단계는 모델이 학습에 사용하지 않은 데이터에서도 긍정/부정 분류를 얼마나 잘 수행하는지 확인하는 과정이다.


In [9]:
# trainer.test는 test_dataloader에서 제공하는 데이터로 최종 성능을 평가합니다.
test_result = trainer.test(model, datamodule=imdb_data)

# 테스트 결과 딕셔너리를 출력하여 test_loss와 test_acc를 확인합니다.
print(test_result)


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │     0.722000002861023     │
│         test_loss         │    0.5895406007766724     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.5895406007766724, 'test_acc': 0.722000002861023}]


## 10. 새로운 리뷰 문장 예측 함수

학습한 Transformer 모델을 사용하여 직접 입력한 영어 영화 리뷰가 긍정인지 부정인지 예측한다. IMDB 데이터셋은 영어 단어 인덱스를 사용하므로 입력 문장도 간단한 영어 리뷰로 넣어 확인한다.


In [10]:
# predict_review 함수는 새로운 리뷰 문장을 입력받아 긍정/부정 예측 결과를 반환합니다.
def predict_review(review_text: str, model: IMDBTransformer, data_module: IMDBDataLoader) -> str:
    # 모델을 평가 모드로 전환하여 Dropout/BatchNorm 등이 평가 방식으로 동작하게 합니다.
    model.eval()

    # 입력 문장을 소문자로 바꾸고 공백 기준으로 단어를 나눕니다.
    words = review_text.lower().split()

    # 각 단어를 IMDB 단어 사전의 정수 ID로 바꿉니다.
    # 사전에 없는 단어는 <UNK>에 해당하는 2번 ID로 처리합니다.
    token_ids = [data_module.word2idx.get(word, 2) for word in words]

    # 모델의 단어 사전 크기를 넘는 토큰은 <UNK>로 바꾸어 임베딩 범위 오류를 방지합니다.
    token_ids = [token_id if token_id < data_module.num_words else 2 for token_id in token_ids]

    # 입력 토큰 시퀀스를 학습 때와 같은 길이로 padding/truncating 처리합니다.
    padded = pad_sequences([token_ids], maxlen=data_module.max_seq_len, value=0, padding='pre', truncating='pre')

    # NumPy 배열을 PyTorch LongTensor로 변환합니다.
    x = torch.LongTensor(padded)

    # 입력 텐서를 모델이 위치한 장치로 이동합니다.
    x = x.to(model.device)

    # 예측 과정에서는 기울기 계산이 필요 없으므로 torch.no_grad()를 사용합니다.
    with torch.no_grad():
        # 모델에 입력을 넣어 클래스별 점수 logits를 계산합니다.
        logits = model(x)

        # softmax를 적용하여 클래스별 확률로 변환합니다.
        probs = torch.softmax(logits, dim=1)

        # 가장 높은 확률을 가진 클래스를 예측값으로 선택합니다.
        pred = torch.argmax(probs, dim=1).item()

    # 예측 클래스가 1이면 긍정, 0이면 부정으로 해석합니다.
    label = 'POSITIVE' if pred == 1 else 'NEGATIVE'

    # 긍정 클래스 확률을 추출합니다.
    positive_prob = probs[0, 1].item()

    # 최종 예측 결과 문자열을 반환합니다.
    return f'예측 결과: {label}, 긍정 확률: {positive_prob:.4f}'

# 긍정적인 의미의 예시 리뷰를 모델에 입력하여 예측 결과를 확인합니다.
print(predict_review('this movie was great and very interesting', model, imdb_data))

# 부정적인 의미의 예시 리뷰를 모델에 입력하여 예측 결과를 확인합니다.
print(predict_review('this movie was boring and terrible', model, imdb_data))


예측 결과: POSITIVE, 긍정 확률: 0.9987
예측 결과: NEGATIVE, 긍정 확률: 0.4389
